# Lexsi GRPO rollout diagnostics

Local machine: Tiny Aya runs on the RTX 5060 Ti (16 GB). The RTX 4060 (8 GB) is too small and is never selected. PyTorch lists the 4060 as `cuda:0` and the 5060 Ti as `cuda:1`, so the live cell masks the process to the 5060 Ti UUID `134c34ef-c155-b001-4e3a-b9134b072c21` instead of trusting the index.

Colab is diagnostic-only when that card is unavailable. `--allow-any-gpu` is required there and must not be used for claimed study evidence.

See `docs/COLAB_ROLLOUT_DIAGNOSTICS.md`.

## 0. Runtime

Runtime → Change runtime type → **GPU** (T4 is enough for a few Tiny Aya bf16 rollouts).

In [ ]:
# Clone + thin train deps on Colab. Locally this worktree and the existing train env are used.
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO = Path("/content/Hackathons")
    if not (REPO / "lexsi").exists():
        subprocess.check_call([
            "git", "clone", "--depth", "1",
            "https://github.com/AriCommits/Hackathons.git", str(REPO),
        ])
    os.chdir(REPO / "lexsi")
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "-e", ".[train]"])
else:
    root = Path.cwd()
    if root.name == "notebooks" and (root.parent / "scripts" / "diagnose_grpo_rollouts.py").exists():
        os.chdir(root.parent)
    elif root.name != "lexsi":
        raise SystemExit(f"run this notebook from the lexsi directory, cwd={root}")
print("cwd", Path.cwd(), "colab", IN_COLAB)

In [ ]:
# HF secret (Colab: Secrets panel -> HF_TOKEN). Local runs use the existing HF login.
import os
from pathlib import Path

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    os.environ["HF_TOKEN"] = token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = token
    print("HF_TOKEN set:", bool(token))
except ImportError:
    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    token_file = Path.home() / ".cache" / "huggingface" / "token"
    print("HF_TOKEN set:", bool(token) or token_file.is_file())

## 1–3. CPU checks (no GPU, no adapter)

- Prompt train vs eval diff
- Reward ledger meaning for −0.1
- Fixture multi-turn dumps (includes one parse_error → −0.1)

**Code finding:** training rollouts are multi-turn (`BehaviorToolRollout`), same as eval. −0.1 is first-decision failure (k=0, invalid_attempts=1), not single-turn TRL.

In [ ]:
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    fixture_out = Path("/content/fixture_rollouts.json")
except ImportError:
    # Windows can have a C:\content directory, so Path("/content").exists() is not Colab.
    fixture_out = Path("results/grpo_diag/fixture_rollouts.json")
    fixture_out.parent.mkdir(parents=True, exist_ok=True)

py = sys.executable
subprocess.check_call([py, "scripts/diagnose_grpo_rollouts.py", "--prompt-diff"])
subprocess.check_call([py, "scripts/diagnose_grpo_rollouts.py", "--explain-reward"])
subprocess.check_call([
    py, "scripts/diagnose_grpo_rollouts.py", "--fixture-rollouts",
    "--count", "4", "--output", str(fixture_out),
])

## 4. Live GPU dumps (Tiny Aya ± e2 adapter)

Locally the adapter is `results/sft_v2/e2/checkpoint-150`. nvidia-smi index 0 is the 5060 Ti, but `CUDA_VISIBLE_DEVICES` follows PyTorch order (4060 = 0, 5060 Ti = 1). The child process therefore sets `CUDA_VISIBLE_DEVICES` to `GPU-` plus the 5060 Ti UUID, then the diagnostic script checks that UUID again. `--allow-any-gpu` is not passed on this machine.

On Colab, upload that checkpoint to `/content/e2-checkpoint-150`. Colab does not have the lab UUID, so that path passes `--allow-any-gpu`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

LAB_5060_UUID = "134c34ef-c155-b001-4e3a-b9134b072c21"

def nvidia_confirms_5060(uuid):
    completed = subprocess.run(
        ["nvidia-smi", "--query-gpu=index,uuid,name", "--format=csv,noheader"],
        check=True, capture_output=True, text=True,
    )
    target = uuid.lower().removeprefix("gpu-")
    for line in completed.stdout.splitlines():
        if not line.strip():
            continue
        _index, raw_uuid, name = [part.strip() for part in line.split(",", 2)]
        if raw_uuid.lower().removeprefix("gpu-") == target and "RTX 5060 Ti" in name:
            return True
    return False

try:
    import google.colab  # noqa: F401
    in_colab = True
except ImportError:
    # Windows can have a C:\content directory, so Path("/content").exists() is not Colab.
    in_colab = False
if in_colab:
    ADAPTER = "/content/e2-checkpoint-150"
    live_out = Path("/content/live_rollouts.json")
else:
    ADAPTER = "results/sft_v2/e2/checkpoint-150"
    live_out = Path("results/grpo_diag/live_rollouts.json")
    live_out.parent.mkdir(parents=True, exist_ok=True)

adapter_args = ["--adapter", ADAPTER] if Path(ADAPTER).is_dir() else []
print("adapter", ADAPTER, "exists", Path(ADAPTER).is_dir())
cmd = [
    sys.executable, "scripts/diagnose_grpo_rollouts.py", "--live-rollouts",
    "--count", "4", "--output", str(live_out), *adapter_args,
]
env = os.environ.copy()
env.pop("LEXSI_ALLOW_ANY_GPU", None)
if in_colab:
    cmd.append("--allow-any-gpu")
else:
    if not nvidia_confirms_5060(LAB_5060_UUID):
        raise SystemExit("RTX 5060 Ti is not visible; refusing to load Tiny Aya on the 4060")
    # Ordinals are not interchangeable: nvidia-smi 0 is the 5060 Ti, but
    # CUDA_VISIBLE_DEVICES=0 is the 4060. The GPU-UUID form pins the 5060 Ti.
    env["CUDA_VISIBLE_DEVICES"] = "GPU-" + LAB_5060_UUID
    print("CUDA_VISIBLE_DEVICES", env["CUDA_VISIBLE_DEVICES"], "uuid", LAB_5060_UUID)
subprocess.check_call(cmd, env=env)

In [ ]:
import json
from pathlib import Path

try:
    import google.colab  # noqa: F401
    candidates = [
        Path("/content/live_rollouts.json"),
        Path("/content/fixture_rollouts.json"),
    ]
except ImportError:
    candidates = [
        Path("results/grpo_diag/live_rollouts.json"),
        Path("results/grpo_diag/fixture_rollouts.json"),
    ]
path = next(p for p in candidates if p.exists())
print("reading", path)
payload = json.loads(path.read_text())
for i, row in enumerate(payload["rollouts"]):
    ledger = row["ledger"]
    print("=" * 60)
    print(f"rollout {i} reason={row['completion_reason']} tokens={row['completion_token_count']}")
    print("prompt:", row["prompt"][:200])
    print("completion:", row["decoded_completion"][:500])
    print("ledger:", {k: ledger[k] for k in (
        "reward", "invalid_attempts", "verified_success", "completion_reason",
        "duplicate_calls", "excess_calls",
    )})
print("truncation_stats:", payload.get("truncation_stats"))

## Unattended one-shot (close laptop after start)

Run **only this next cell** after Runtime → GPU. It mounts Drive, installs, dumps 4 live rollouts, writes `MyDrive/lexsi/grpo_diag/live_rollouts.json` + a compact summary with a `completion_reason` Counter, then prints `DONE`.

**Caveat:** free Colab may still stop if the browser tab dies. Leave the tab open on a phone if you can; otherwise prefer Colab Pro. There is no email callback — recover results from Drive later.

Optional: put e2 at `MyDrive/lexsi/e2-checkpoint-150/` first. Without it, base Tiny Aya still probes format/truncation/template.

In [ ]:
# Unattended GRPO rollout dump -> Google Drive. Local stepwise cells already ran the same checks.
import json, os, subprocess, sys
from collections import Counter
from pathlib import Path

try:
    from google.colab import drive, userdata
except ImportError:
    print("Local run already executed the stepwise cells. Skipping Drive mount and the duplicate dump.")
else:
    drive.mount("/content/drive", force_remount=False)
    OUT_DIR = Path("/content/drive/MyDrive/lexsi/grpo_diag")
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    OUT = OUT_DIR / "live_rollouts.json"
    SUMMARY = OUT_DIR / "live_rollouts_summary.json"

    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]
    os.environ["LEXSI_ALLOW_ANY_GPU"] = "1"

    REPO = Path("/content/Hackathons")
    if not (REPO / "lexsi").exists():
        subprocess.check_call([
            "git", "clone", "--depth", "1", "-b",
            "cursor/lexsi-colab-rollout-diagnostics-8ccb",
            "https://github.com/AriCommits/Hackathons.git", str(REPO),
        ])
    os.chdir(REPO / "lexsi")
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "-e", ".[train]"])

    ADAPTER = Path("/content/drive/MyDrive/lexsi/e2-checkpoint-150")
    cmd = [
        sys.executable, "scripts/diagnose_grpo_rollouts.py", "--live-rollouts",
        "--allow-any-gpu", "--count", "4", "--output", str(OUT),
    ]
    if ADAPTER.is_dir():
        cmd.extend(["--adapter", str(ADAPTER)])

    print("STARTING unattended dump; safe to close laptop after this line prints.", flush=True)
    subprocess.check_call(cmd)
    payload = json.loads(OUT.read_text())
    reasons = Counter(row["completion_reason"] for row in payload["rollouts"])
    summary = {
        "output": str(OUT),
        "completion_reason_counts": dict(reasons),
        "rewards": [row["ledger"]["reward"] for row in payload["rollouts"]],
        "invalid_attempts": [row["ledger"]["invalid_attempts"] for row in payload["rollouts"]],
        "adapter": str(ADAPTER) if ADAPTER.is_dir() else None,
        "truncation_stats": payload.get("truncation_stats"),
    }
    SUMMARY.write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n")
    print("DONE", summary)
    print("Forward this Counter later:", reasons)

## What still needs the Windows dual-GPU lab

- Full GRPO study with the pinned 5060 Ti UUID and protected `results/rl_v2/*` dirs
- Official memory probe / dual-card isolation claims
- Anything reported as study evidence (not diagnostic)